In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [3]:
df = pd.read_csv(
    "final_data_for_vehicle_forecasting.csv"
)

df['Month'] = pd.to_datetime(
    df['Month'],
    dayfirst=True
)

print(df.shape)
print(df['Month'].min())
print(df['Month'].max())

(49680, 6)
2023-04-01 00:00:00
2025-03-01 00:00:00


In [4]:
def evaluate(actual, forecast):

    actual = np.array(actual)
    forecast = np.array(forecast)

    mae = mean_absolute_error(
        actual,
        forecast
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            forecast
        )
    )

    non_zero_mask = actual != 0

    if non_zero_mask.sum() == 0:

        return mae, rmse, np.nan, np.nan

    mape = (
        np.mean(
            np.abs(
                (
                    actual[non_zero_mask]
                    -
                    forecast[non_zero_mask]
                )
                /
                actual[non_zero_mask]
            )
        )
        * 100
    )

    accuracy = max(
        0,
        100 - mape
    )

    return mae, rmse, mape, accuracy

In [5]:
def moving_average_forecast(train):

    return [train.tail(3).mean()] * 3


def ses_forecast(train):

    model = SimpleExpSmoothing(train)

    fit = model.fit()

    return fit.forecast(3)


def holt_forecast(train):

    model = Holt(train)

    fit = model.fit()

    return fit.forecast(3)


def arima_forecast(train):

    model = ARIMA(
        train,
        order=(1,1,1)
    )

    fit = model.fit()

    return fit.forecast(3)


def sarima_forecast(train):

    model = SARIMAX(
        train,
        order=(1,1,1),
        seasonal_order=(1,1,1,12)
    )

    fit = model.fit(
        disp=False
    )

    return fit.forecast(3)

In [6]:
forecast_models = {

    'Moving Average':
        moving_average_forecast,

    'SES':
        ses_forecast,

    'Holt':
        holt_forecast,

    'ARIMA':
        arima_forecast,

    'SARIMA':
        sarima_forecast
}

In [7]:
results = []

In [8]:
for halb in df['Halb'].unique():

    df_halb = df[
        df['Halb'] == halb
    ]

    for engine in df_halb['engine_type'].unique():

        df_engine = df_halb[
            df_halb['engine_type'] == engine
        ]

        for model in df_engine['map_model'].unique():

            df_model = df_engine[
                df_engine['map_model'] == model
            ]

            for vehicle in df_model[
                'Vehicle_Type'
            ].unique():

                temp = df_model[
                    df_model['Vehicle_Type']
                    == vehicle
                ]

                series = (
                    temp.groupby('Month')
                    ['total_demand']
                    .sum()
                    .sort_index()
                )

                train = series.loc[
                    :'2024-12-01'
                ]

                test = series.loc[
                    '2025-01-01':'2025-03-01'
                ]

                if len(train) < 12:
                    continue

                if len(test) != 3:
                    continue

                for model_name, model_func in (
                    forecast_models.items()
                ):

                    try:

                        forecast = model_func(
                            train
                        )

                        mae, rmse, mape, acc = (
                            evaluate(
                                test,
                                forecast
                            )
                        )

                        results.append([

                            halb,
                            engine,
                            model,
                            vehicle,

                            model_name,

                            mae,
                            rmse,
                            mape,
                            acc

                        ])

                    except:

                        continue

/Users/utkarshraj/Desktop/Internship/Vehicle Demand Forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/utkarshraj/Desktop/Internship/Vehicle Demand Forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/utkarshraj/Desktop/Internship/Vehicle Demand Forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/utkarshraj/Desktop/Internship/Vehicle Demand Forecasting/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency M

In [9]:
results_df = pd.DataFrame(

    results,

    columns=[

        'Halb',
        'Engine',
        'Model',
        'Vehicle_Type',

        'Forecast_Model',

        'MAE',
        'RMSE',
        'MAPE',
        'Accuracy'
    ]
)

In [10]:
print(results_df.shape)

print(
    results_df[
        'Forecast_Model'
    ].value_counts()
)

(10350, 9)
Forecast_Model
Moving Average    2070
SES               2070
Holt              2070
ARIMA             2070
SARIMA            2070
Name: count, dtype: int64


In [11]:
model_summary = (

    results_df
    .groupby(
        'Forecast_Model'
    )[
        ['MAE',
         'RMSE',
         'MAPE',
         'Accuracy']
    ]
    .mean()

    .sort_values(
        'Accuracy',
        ascending=False
    )
)

model_summary

,MAE,RMSE,MAPE,Accuracy
Forecast_Model,,,,
ARIMA,3.789408,4.544382,94.872078,26.374195
SES,3.396911,4.182625,90.578277,24.458292
Moving Average,3.473698,4.250268,94.300064,22.846288
Holt,3.681303,4.462875,99.074444,22.120181
SARIMA,6.059157,7.242885,140.840742,17.232838


In [15]:
df_ml = df.copy()

df_ml = df_ml.sort_values(
    [
        'Halb',
        'engine_type',
        'map_model',
        'Vehicle_Type',
        'Month'
    ]
)

In [16]:
df_ml['Lag_1'] = (
    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']
    .shift(1)
)

df_ml['Lag_2'] = (
    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']
    .shift(2)
)

df_ml['Lag_3'] = (
    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']
    .shift(3)
)

df_ml['Lag_6'] = (
    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']
    .shift(6)
)

In [17]:
df_ml['Rolling_Mean_3'] = (

    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']

    .transform(
        lambda x:
        x.shift(1)
        .rolling(3)
        .mean()
    )
)

df_ml['Rolling_Mean_6'] = (

    df_ml.groupby(
        [
            'Halb',
            'engine_type',
            'map_model',
            'Vehicle_Type'
        ]
    )['total_demand']

    .transform(
        lambda x:
        x.shift(1)
        .rolling(6)
        .mean()
    )
)

In [18]:
df_ml['Month_Number'] = (
    df_ml['Month']
    .dt.month
)

df_ml['Quarter'] = (
    df_ml['Month']
    .dt.quarter
)

df_ml['Year'] = (
    df_ml['Month']
    .dt.year
)

In [19]:
df_ml = df_ml.dropna()

print(df_ml.shape)

(37260, 15)


In [20]:
df_ml = pd.get_dummies(

    df_ml,

    columns=[
        'Halb',
        'engine_type',
        'map_model',
        'Vehicle_Type'
    ]
)

In [21]:
train_ml = df_ml[
    df_ml['Month']
    <= '2024-12-01'
]

test_ml = df_ml[
    df_ml['Month']
    >= '2025-01-01'
]

In [22]:
X_train = train_ml.drop(

    columns=[
        'Month',
        'total_demand'
    ]
)

y_train = train_ml[
    'total_demand'
]

X_test = test_ml.drop(

    columns=[
        'Month',
        'total_demand'
    ]
)

y_test = test_ml[
    'total_demand'
]

In [23]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(
    X_train,
    y_train
)

lr_pred = lr_model.predict(
    X_test
)

lr_results = evaluate(
    y_test,
    lr_pred
)

print(lr_results)

(3.857277050123909, np.float64(15.774518073429784), np.float64(94.2048605732891), np.float64(5.795139426710904))


In [24]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(

    n_estimators=200,

    max_depth=10,

    random_state=42,

    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(
    X_test
)

rf_results = evaluate(
    y_test,
    rf_pred
)

print(rf_results)

(3.720705940431947, np.float64(16.34336038462475), np.float64(78.91749964384637), np.float64(21.08250035615363))


In [25]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=6,

    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(
    X_test
)

xgb_results = evaluate(
    y_test,
    xgb_pred
)

print(xgb_results)

(3.636625289916992, np.float64(16.0490343225238), np.float64(82.27376793858392), np.float64(17.726232061416084))


In [26]:
ml_results = pd.DataFrame({

    'Model': [

        'Linear Regression',

        'Random Forest',

        'XGBoost'

    ],

    'MAE': [

        lr_results[0],

        rf_results[0],

        xgb_results[0]

    ],

    'RMSE': [

        lr_results[1],

        rf_results[1],

        xgb_results[1]

    ],

    'MAPE': [

        lr_results[2],

        rf_results[2],

        xgb_results[2]

    ],

    'Accuracy': [

        lr_results[3],

        rf_results[3],

        xgb_results[3]

    ]
})

ml_results.sort_values(
    'Accuracy',
    ascending=False
)

,Model,MAE,RMSE,MAPE,Accuracy
1,Random Forest,3.720706,16.343360,78.917500,21.082500
2,XGBoost,3.636625,16.049034,82.273768,17.726232
0,Linear Regression,3.857277,15.774518,94.204861,5.795139


In [27]:
df.to_excel("forecast_results_with_ML.xlsx", index=False)